# Trénink sítě na přestavovače brzdy (Výkaz vozů)

Z fotek, které vozmistři pořídili v aplikaci **Výkaz vozů**, natrénuje malou neuronovou síť (MobileNetV3) pro každý přestavovač:

| druh | polohy |
|---|---|
| `rezim` | G / P |
| `naklad` | prázdný / ložený / samočinný |
| `kohout` | brzda zapnuta / vypnuta |
| `rucni` | ruční brzda odbrzděna / zabrzděna / nemá |
| `blocks` | druh brzdových špalíků K / L / LL / D / F |
| `axles` | počet náprav 2 / 3 / 4 / 6 / 8 |

**Postup**
1. V aplikaci: *Data pro trénink → Exportovat nové*, ZIP uložit na Google Disk do složky `Vykaz_vozu_data`.
2. Tady: *Runtime → Change runtime type → T4 GPU* (jde to i bez GPU, jen pomaleji) a *Run all*.
3. Hotové sítě se uloží na Disk do `Vykaz_vozu_modely/` – soubory `*.onnx` a `brzdy.json` zkopírovat do složky `models/` aplikace a vydat novou verzi.

Síť se natrénuje jen u přestavovače, který má aspoň dvě polohy s dostatkem fotek (`MIN_FOTEK`).
Na začátek stačí desítky fotek od každé polohy, spolehlivá síť chce stovky – z různých vozů, za dne i v noci.

In [ ]:
!pip -q install onnx onnxscript onnxruntime

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DISK = Path('/content/drive/MyDrive')
VSTUP = DISK / 'Vykaz_vozu_data'      # sem ukládat ZIPy z aplikace
VYSTUP = DISK / 'Vykaz_vozu_modely'   # sem se zapíšou hotové sítě
DATA = Path('/content/data')

MIN_FOTEK = 8      # nejméně fotek od každé polohy, jinak se přestavovač přeskočí
EPOCHY = 20
VELIKOST = 224     # aplikace zmenšuje celou fotku na čtverec 224 × 224 (bez ořezu) – musí sedět
VYSTUP.mkdir(exist_ok=True)
print('ZIPy:', sorted(p.name for p in VSTUP.glob('*.zip')))

In [ ]:
# Rozbalení všech exportů do jedné složky (stejná fotka ve více exportech se jen přepíše).
import zipfile, csv, io, collections
DATA.mkdir(exist_ok=True)
for z in sorted(VSTUP.glob('*.zip')):
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)

# Popisky bere z labels.csv posledního exportu, kde fotka byla (opravy polohy se tak projeví).
POPISKY = {}
for z in sorted(VSTUP.glob('*.zip')):
    with zipfile.ZipFile(z) as f:
        text = f.read('labels.csv').decode('utf-8-sig')
    for r in csv.DictReader(io.StringIO(text), delimiter=';'):
        POPISKY[r['soubor'].split('/')[-1]] = r

DRUHY = ['rezim', 'naklad', 'kohout', 'rucni', 'blocks', 'axles']
for druh in DRUHY:
    pocty = collections.Counter(r['poloha'] for r in POPISKY.values() if r['druh'] == druh)
    print(f'{druh:8}', dict(pocty) or '–')

In [ ]:
import random, time, json
import numpy as np
import torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms

zarizeni = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Počítá se na:', zarizeni)

NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# Bez zrcadlení: strana, na které páka je, určuje polohu.
TRENINK = transforms.Compose([
    transforms.Resize((VELIKOST, VELIKOST)),
    transforms.RandomAffine(degrees=12, translate=(0.06, 0.06), scale=(0.85, 1.15)),
    transforms.ColorJitter(0.5, 0.5, 0.4, 0.03),
    transforms.RandomGrayscale(0.15),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
    transforms.ToTensor(), NORM,
])
KONTROLA = transforms.Compose([transforms.Resize((VELIKOST, VELIKOST)), transforms.ToTensor(), NORM])

class Fotky(torch.utils.data.Dataset):
    def __init__(self, polozky, tf):
        self.polozky, self.tf = polozky, tf
    def __len__(self):
        return len(self.polozky)
    def __getitem__(self, i):
        cesta, trida = self.polozky[i]
        return self.tf(Image.open(cesta).convert('RGB')), trida

def fotky_druhu(druh):
    # aktuální poloha z labels.csv; soubor leží ve složce podle polohy v době exportu
    polozky = collections.defaultdict(list)
    for jmeno, r in POPISKY.items():
        if r['druh'] != druh:
            continue
        nalezene = list((DATA / druh).glob(f'*/{jmeno}'))
        if nalezene:
            polozky[r['poloha']].append(nalezene[0])
    return polozky

In [ ]:
def trenuj(druh):
    polozky = {k: v for k, v in fotky_druhu(druh).items() if len(v) >= MIN_FOTEK}
    if len(polozky) < 2:
        print(f'{druh}: málo fotek, přeskakuji')
        return None
    tridy = sorted(polozky)
    random.seed(1)
    trenink, kontrola = [], []
    for t, cesty in polozky.items():
        cesty = sorted(cesty)
        random.shuffle(cesty)
        n = max(2, len(cesty) // 5)                      # 20 % na kontrolu
        kontrola += [(c, tridy.index(t)) for c in cesty[:n]]
        trenink += [(c, tridy.index(t)) for c in cesty[n:]]
    print(f'{druh}: třídy {tridy}, trénink {len(trenink)}, kontrola {len(kontrola)}')

    sit = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    sit.classifier[3] = nn.Linear(sit.classifier[3].in_features, len(tridy))
    sit = sit.to(zarizeni)
    # vyrovnání nestejného počtu fotek mezi polohami
    pocty = np.bincount([t for _, t in trenink], minlength=len(tridy))
    vahy = torch.tensor(len(trenink) / (len(tridy) * np.maximum(pocty, 1)), dtype=torch.float32, device=zarizeni)
    ztrata = nn.CrossEntropyLoss(weight=vahy, label_smoothing=0.05)
    opt = torch.optim.AdamW(sit.parameters(), lr=5e-4, weight_decay=1e-4)
    plan = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHY)
    dl_t = torch.utils.data.DataLoader(Fotky(trenink, TRENINK), batch_size=32, shuffle=True, num_workers=2)
    dl_k = torch.utils.data.DataLoader(Fotky(kontrola, KONTROLA), batch_size=64, num_workers=2)

    nejlepsi, stav = -1, None
    for ep in range(EPOCHY):
        sit.train()
        for x, y in dl_t:
            opt.zero_grad()
            ztrata(sit(x.to(zarizeni)), y.to(zarizeni)).backward()
            opt.step()
        plan.step()
        sit.eval()
        odhady = []
        with torch.no_grad():
            for x, _ in dl_k:
                odhady += sit(x.to(zarizeni)).argmax(1).cpu().tolist()
        presnost = np.mean([o == t for o, (_, t) in zip(odhady, kontrola)])
        if presnost >= nejlepsi:
            nejlepsi, stav = presnost, {k: v.detach().clone() for k, v in sit.state_dict().items()}
        print(f'  epocha {ep + 1:2}: přesnost na kontrolních fotkách {presnost:.0%}')
    sit.load_state_dict(stav)
    sit.eval()

    # chybně poznané kontrolní fotky – často odhalí špatně zadanou polohu
    with torch.no_grad():
        for cesta, t in kontrola:
            o = sit(KONTROLA(Image.open(cesta).convert('RGB'))[None].to(zarizeni)).argmax(1).item()
            if o != t:
                print(f'  ✗ {cesta.name}: zadáno {tridy[t]}, síť říká {tridy[o]}')
    print(f'{druh}: nejlepší přesnost {nejlepsi:.0%}')
    return sit.cpu(), tridy, nejlepsi

In [ ]:
import onnxruntime as ort

razitko = time.strftime('%Y%m%d_%H%M')
konfig_cesta = VYSTUP / 'brzdy.json'
konfig = json.loads(konfig_cesta.read_text()) if konfig_cesta.exists() else {}

for druh in DRUHY:
    vysledek = trenuj(druh)
    if not vysledek:
        continue
    sit, tridy, presnost = vysledek
    soubor = f'{druh}_{razitko}.onnx'   # nové jméno = aplikace si síť určitě stáhne znovu
    vzor = torch.randn(1, 3, VELIKOST, VELIKOST)
    try:
        torch.onnx.export(sit, vzor, VYSTUP / soubor, input_names=['obraz'], output_names=['logity'],
                          opset_version=17, dynamo=False)
    except TypeError:   # starší PyTorch nezná parametr dynamo
        torch.onnx.export(sit, vzor, VYSTUP / soubor, input_names=['obraz'], output_names=['logity'], opset_version=17)
    # kontrola: ONNX musí počítat stejně jako PyTorch
    sez = ort.InferenceSession(str(VYSTUP / soubor))
    rozdil = np.abs(sez.run(None, {'obraz': vzor.numpy()})[0] - sit(vzor).detach().numpy()).max()
    print(f'{soubor}: rozdíl ONNX × PyTorch {rozdil:.2e}')
    konfig[druh] = {'file': soubor, 'classes': tridy, 'size': VELIKOST,
                    'accuracy': round(float(presnost), 3), 'trained': razitko}

konfig_cesta.write_text(json.dumps(konfig, ensure_ascii=False, indent=2))
print(json.dumps(konfig, ensure_ascii=False, indent=2))
print('Do složky models/ aplikace zkopírujte brzdy.json a nové soubory .onnx z', VYSTUP)